In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)
os.getenv("OPENAI_API_KEY")

In [ ]:
os.getenv("TAVILY_API_KEY")

In [ ]:
from typing import Annotated, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

In [ ]:
model_low_temp = ChatOpenAI(
    model='gpt-5-nano', 
    temperature=0, 
)

model_high_temp = ChatOpenAI(
    temperature=0.7, 
)

In [ ]:
class State(TypedDict):
    messages:Annotated[list, add_messages]
    user_query:str
    sql_query:str
    sql_explanation:str

In [ ]:
general_prompt = SystemMessage(
    content = """당신은 친절한 데이터 분석가입니다.
    사용자의 질문을 바탕으로 SQL쿼리를 작성하세요.
    """
)

In [ ]:
def generate_sql(state: State) -> State:
    user_message = HumanMessage(state['user_query'])
    
    # generate_prompt -> general_sql 로 수정
    messages = [general_sql, *state['messages'], user_message] 
    
    res = model_low_temp.invoke(messages)
    return {
        'sql_query': res.content,
        # 대화 기록 업데이트
        'messages': [user_message, res],
    }

In [ ]:
explain_prompt = SystemMessage(
    content = "당신은 친절한 데이터 분석가입니다. 사용자에게 SQL쿼리를 친절하게 설명하세요."
)

In [ ]:
def explain_sql(state: State) -> State:
    messages = [
        explain_prompt,
        # 이전 단계의 사용자의 질문과 SQL 쿼리
        *state['messages'],
    ]
    res = model_high_temp.invoke(messages)
    return {
        'sql_explanation': res.content,
        # 대화 기록 업데이트
        'messages': [res],
    }

In [ ]:
builder = StateGraph(State)
builder.add_node('generate_sql', generate_sql)
builder.add_node('explain_sql', explain_sql)

builder.add_edge(START, 'generate_sql')
builder.add_edge('generate_sql', 'explain_sql')
builder.add_edge('explain_sql', END)

graph = builder.compile()

In [ ]:
from IPython.display import display, Markdown

# Get the Mermaid code
mermaid_code = graph.get_graph().draw_mermaid()

# Display in Jupyter
display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

In [ ]:
graph

In [ ]:
response = graph.invoke({
    'user_query': "각 품목의 판매량을 구해주세요."
})

for msg in response["messages"]:
    msg.pretty_print()

# SQL Database Toolkit

In [ ]:
import sqlite3
from langchain_community.utilities.sql_database import SQLDatabase
from sqlalchemy import create_engine

db_path = "sqlite:///chinook/chinook.db"

engine = create_engine(
    db_path,
    connect_args = {"check_same_thread": False}
)

In [ ]:
db = SQLDatabase(engine)
db.get_usable_table_names()

In [ ]:
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit

llm = ChatOpenAI(
    model = "gpt-5-nano",
    temperature= 0.7
)

toolkit = SQLDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()

In [ ]:
tools

pip install langchainhub

In [ ]:
from langchain_classic import hub

prompt_template = hub.pull("langchain-ai/sql-agent-system-prompt")

In [ ]:
from langchain.agents import create_agent

system_prompt = prompt_template.format(dialect="SQLite", top_k = 3)

agent = create_agent(
    llm, 
    tools,
    system_prompt = system_prompt
)

In [ ]:
query = "어느 나라의 고객들이 가장 많이 소비합니까?"
response = agent.invoke(
    {"messages": [("user",query)]}
)

In [ ]:
for msg in response["messages"]:
    msg.pretty_print()

In [ ]:
query = "전세계 탑3의 축구클렁을 알려줘."

In [ ]:
events = agent.stream(
    {"messages":[("user",query)]}, stream_mode = "values"
)

for event in events:
    event['messages'][-1].pretty_print()

In [ ]:
response = agent.invoke(
    {"messages": [("user",query)]}
)

for msg in response["messages"]:
    msg.pretty_print()